In [1]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from time import time

from sklearn.model_selection import (
    StratifiedKFold, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    BaggingClassifier,
    StackingClassifier,
)
import xgboost as xgb
import lightgbm as lgb
import pickle
warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
REPO_ROOT    = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR     = os.path.join(REPO_ROOT, 'Datasets_all')
OUT_DIR      = Path('models')
OUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_SPLITS     = 5
CHAMPION_F1  = 0.6484   # Score from A5b

In [3]:
movement_features_df = pd.read_csv(os.path.join(DATA_DIR, 'aimoscores.csv'))
weaklink_scores_df   = pd.read_csv(os.path.join(DATA_DIR, 'scores_and_weaklink.csv'))

print('Movement features shape:', movement_features_df.shape)
print('Weak link scores shape:', weaklink_scores_df.shape)

DUPLICATE_NASM_COLS = [
    'No_1_NASM_Deviation',
    'No_2_NASM_Deviation',
    'No_3_NASM_Deviation',
    'No_4_NASM_Deviation',
    'No_5_NASM_Deviation',
]

movement_features_df = movement_features_df.drop(columns=DUPLICATE_NASM_COLS)
print('Shape after duplicate removal:', movement_features_df.shape)

weaklink_categories = [
    'ExcessiveForwardLean', 'ForwardHead', 'LeftArmFallForward',
    'LeftAsymmetricalWeightShift', 'LeftHeelRises', 'LeftKneeMovesInward',
    'LeftKneeMovesOutward', 'LeftShoulderElevation', 'RightArmFallForward',
    'RightAsymmetricalWeightShift', 'RightHeelRises', 'RightKneeMovesInward',
    'RightKneeMovesOutward', 'RightShoulderElevation',
]

weaklink_scores_df['WeakestLink'] = (
    weaklink_scores_df[weaklink_categories].idxmax(axis=1)
)
print('Weakest Link class distribution:')
print(weaklink_scores_df['WeakestLink'].value_counts())

Movement features shape: (2094, 43)
Weak link scores shape: (2096, 17)
Shape after duplicate removal: (2094, 38)
Weakest Link class distribution:
WeakestLink
LeftArmFallForward              616
RightArmFallForward             458
RightKneeMovesOutward           274
RightShoulderElevation          245
ExcessiveForwardLean            128
ForwardHead                     109
LeftAsymmetricalWeightShift      80
LeftShoulderElevation            55
LeftKneeMovesOutward             54
RightKneeMovesInward             45
RightAsymmetricalWeightShift     20
LeftHeelRises                     7
LeftKneeMovesInward               3
RightHeelRises                    2
Name: count, dtype: int64


In [4]:
# Merge Datasets
target_df = weaklink_scores_df[['ID', 'WeakestLink']].copy()
merged_df = movement_features_df.merge(target_df, on='ID', how='inner')
print('Merged dataset shape:', merged_df.shape)

EXCLUDE_COLS    = ['ID', 'WeakestLink', 'EstimatedScore']
feature_columns = [c for c in merged_df.columns if c not in EXCLUDE_COLS]

X = merged_df[feature_columns].values
y = merged_df['WeakestLink'].values

print(f'Feature matrix shape : {X.shape}')
print(f'Number of features   : {len(feature_columns)}')
print(f'Number of classes    : {len(np.unique(y))}')

Merged dataset shape: (2094, 39)
Feature matrix shape : (2094, 36)
Number of features   : 36
Number of classes    : 14


In [9]:
C_range     = [2**i for i in range(-5, 10, 4)]
gamma_range = [2**i for i in range(-10, 4, 4)]

svm_param_grid = [
    {'svm__kernel': ['rbf'],    'svm__C': C_range, 'svm__gamma': gamma_range, 'svm__class_weight': ['balanced']},
    {'svm__kernel': ['poly'],   'svm__C': C_range, 'svm__gamma': gamma_range, 'svm__degree': [2, 3], 'svm__class_weight': ['balanced']},
    {'svm__kernel': ['linear'], 'svm__C': C_range, 'svm__class_weight': ['balanced']},
]

In [10]:
outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# Pipeline keeps scaler inside each fold
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(probability=True, random_state=RANDOM_STATE)),
])

nested_svm = GridSearchCV(
    estimator  = svm_pipeline,
    param_grid = svm_param_grid,
    cv         = inner_cv,
    scoring    = 'f1_weighted',
    n_jobs     = -1,
    verbose    = 0,
    refit      = True,
)
nested_svm_scores = cross_val_score(
    nested_svm, X, y,
    cv      = outer_cv,
    scoring = 'f1_weighted',
    n_jobs  = -1,
)

print(f'Per-fold F1 : {np.round(nested_svm_scores, 4)}')
print(f'Mean F1     : {nested_svm_scores.mean():.4f} +/- {nested_svm_scores.std():.4f}')

Per-fold F1 : [0.5938 0.5981 0.5761 0.6399 0.6123]
Mean F1     : 0.6040 +/- 0.0213


In [11]:

soft_voting = VotingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5, min_samples_leaf=2, class_weight='balanced_subsample',
                                       random_state=RANDOM_STATE, n_jobs=-1)),
        ('lr',  LogisticRegression( max_iter=1000, class_weight='balanced',random_state=RANDOM_STATE)),
        ('xgb', xgb.XGBClassifier(  n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8,
                                    colsample_bytree=0.8, random_state=RANDOM_STATE,class_weight='balanced', n_jobs=-1 )),
        ('lgb', lgb.LGBMClassifier( n_estimators=200, learning_rate=0.1, class_weight='balanced',subsample=0.8, colsample_bytree=0.8,
                                    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1 )),
        ('knn', KNeighborsClassifier(n_neighbors=7)),
        ('lda', LinearDiscriminantAnalysis()),
    ],
    voting='soft',
    n_jobs=-1,
)
sv_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('voting', soft_voting),
])

print('Running CV for Soft Voting champion')
sv_scores = cross_val_score(sv_pipeline, X, y, cv=outer_cv, scoring='f1_weighted', n_jobs=-1)
print(f'Per-fold F1 : {np.round(sv_scores, 4)}')
print(f'Mean F1     : {sv_scores.mean():.4f} +/- {sv_scores.std():.4f}')

Running CV for Soft Voting champion
Per-fold F1 : [0.6316 0.6433 0.6289 0.7063 0.6331]
Mean F1     : 0.6486 +/- 0.0292


In [12]:
CHAMPION_F1 = 0.6484  # A5b reported score

results = [
    {'Model': 'SVM (Nested CV)',           'F1_mean': nested_svm_scores.mean(), 'F1_std': nested_svm_scores.std(), '_scores': nested_svm_scores},
    {'Model': 'A5 Champion (Soft Voting)', 'F1_mean': sv_scores.mean(),         'F1_std': sv_scores.std(),         '_scores': sv_scores},
]

results_df = pd.DataFrame([{k:v for k,v in r.items() if k != '_scores'} for r in results])
results_df = results_df.sort_values('F1_mean', ascending=False).reset_index(drop=True)
results_df['vs_A5b'] = results_df['F1_mean'].apply(lambda f: f'{(f - CHAMPION_F1)/CHAMPION_F1*100:+.1f}%')
print(results_df[['Model','F1_mean','F1_std','vs_A5b']].to_string(index=False))

                    Model  F1_mean   F1_std vs_A5b
A5 Champion (Soft Voting) 0.648627 0.029224  +0.0%
          SVM (Nested CV) 0.604041 0.021310  -6.8%


In [13]:
from scipy import stats

def corrected_resampled_ttest(scores_a, scores_b, n_train, n_test):
    k       = len(scores_a)
    diff    = scores_a - scores_b
    d_bar   = diff.mean()
    s_sq    = diff.var(ddof=1)
    var_corr = (1/k + n_test/n_train) * s_sq
    t_stat  = d_bar / np.sqrt(var_corr)
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=k-1))
    return float(t_stat), float(p_value)

n_total      = len(X)
n_test_fold  = n_total // N_SPLITS
n_train_fold = n_total - n_test_fold

score_map = {r['Model']: r['_scores'] for r in results}
sv_f1     = score_map['A5 Champion (Soft Voting)']
svm_f1    = score_map['SVM (Nested CV)']

t, p = corrected_resampled_ttest(svm_f1, sv_f1, n_train_fold, n_test_fold)
sig  = 'Significant' if p < 0.05 else 'Not significant'
print(f'SVM (Nested CV) vs A5 Champion: t={t:+.3f}, p={p:.4f}  -> {sig}')

SVM (Nested CV) vs A5 Champion: t=-3.913, p=0.0173  -> Significant


In [14]:
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(probability=True, random_state=RANDOM_STATE)),
])

final_grid = GridSearchCV(
    final_pipeline, svm_param_grid,
    cv      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
    scoring = 'f1_weighted',
    n_jobs  = -1, verbose=1,
)
final_grid.fit(X, y)
print(f'Best params: {final_grid.best_params_}')

with open(OUT_DIR / 'champion_svm.pkl', 'wb') as f:
    pickle.dump(final_grid.best_estimator_, f)
print('Model saved to champion_svm.pkl')

Fitting 5 folds for each of 52 candidates, totalling 260 fits
Best params: {'svm__C': 8, 'svm__class_weight': 'balanced', 'svm__gamma': 0.015625, 'svm__kernel': 'rbf'}
Model saved to champion_svm.pkl
